# Lab Assignment 6 — 20 Newsgroups Text Classification with W&B Logging

Trains an Embedding + Conv1D classifier on the 20 Newsgroups dataset and logs metrics, sample predictions, confusion matrix, and model artifacts to Weights & Biases.

## 1. Install Dependencies

In [1]:
!pip install wandb scikit-learn tensorflow -q

## 2. Imports

In [2]:
import os
import numpy as np
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

from sklearn.datasets import fetch_20newsgroups
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow import keras as k
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [3]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pritammane105 (pritammane105-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Custom Callbacks

In [4]:
class LogLRCallback(k.callbacks.Callback):
    """Log the optimizer learning rate at the end of each epoch."""
    def on_epoch_end(self, epoch, logs=None):
        lr = self.model.optimizer.learning_rate
        lr_val = float(lr.numpy() if hasattr(lr, 'numpy') else lr)
        wandb.log({'lr': lr_val}, step=self.model.optimizer.iterations.numpy())


class LogSamplesCallback(k.callbacks.Callback):
    """
    Log a W&B table of text snippets with true/predicted newsgroup labels each epoch.
    """
    def __init__(self, x, y, texts, label_names, max_rows=32):
        super().__init__()
        self.x = x[:max_rows]
        self.y = y[:max_rows]
        self.texts = texts[:max_rows]
        self.label_names = label_names

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x, verbose=0)
        y_true = np.argmax(self.y, axis=1)
        y_pred = np.argmax(preds, axis=1)

        table = wandb.Table(columns=['text_snippet', 'true_label', 'predicted_label', 'correct', 'confidence'])
        for i in range(len(self.x)):
            snippet = self.texts[i][:120] + '...'  # truncate for readability
            table.add_data(
                snippet,
                self.label_names[y_true[i]],
                self.label_names[y_pred[i]],
                bool(y_true[i] == y_pred[i]),
                float(np.max(preds[i])),
            )
        wandb.log({f'samples/epoch_{epoch + 1}': table})


class ConfusionMatrixCallback(k.callbacks.Callback):
    """Log a 20x20 confusion matrix on the validation set each epoch."""
    def __init__(self, x_val, y_val, label_names):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.label_names = label_names

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, verbose=0)
        y_true = np.argmax(self.y_val, axis=1)
        y_pred = np.argmax(preds, axis=1)
        cm_plot = wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true.tolist(),
            preds=y_pred.tolist(),
            class_names=self.label_names,
        )
        wandb.log({'confusion_matrix': cm_plot})

## 4. Trainer

In [7]:
class NewsgroupsTrainer:
    def __init__(self, project_name='LabAssignment6-20NG', run_name='conv1d_baseline'):
        self.cfg = dict(
            vocab_size=20000,
            max_len=300,
            embedding_dim=64,
            num_filters=128,
            kernel_size=5,
            dropout=0.3,
            learning_rate=0.001,
            epochs=12,
            batch_size=64,
        )
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=self.cfg,
            settings=wandb.Settings(start_method='thread'),
        )
        self.config = wandb.config
        self._prepare_data()

    def _prepare_data(self):
        print('Loading 20 Newsgroups dataset...')
        train_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
        test_data  = fetch_20newsgroups(subset='test',  remove=('headers', 'footers', 'quotes'))

        self.label_names = train_data.target_names  # 20 class names
        self.num_classes  = len(self.label_names)

        # Tokenize
        tokenizer = Tokenizer(num_words=self.config.vocab_size, oov_token='<OOV>')
        tokenizer.fit_on_texts(train_data.data)

        X_train = pad_sequences(tokenizer.texts_to_sequences(train_data.data),
                                maxlen=self.config.max_len, padding='post', truncating='post')
        X_test  = pad_sequences(tokenizer.texts_to_sequences(test_data.data),
                                maxlen=self.config.max_len, padding='post', truncating='post')

        self.X_train = X_train
        self.X_test  = X_test
        self.y_train = to_categorical(train_data.target, self.num_classes)
        self.y_test  = to_categorical(test_data.target,  self.num_classes)

        # Keep raw texts for the sample table callback
        self.test_texts = test_data.data

        print(f'Train: {X_train.shape}, Test: {X_test.shape}, Classes: {self.num_classes}')

    def _build_model(self):
        inputs = k.Input(shape=(self.config.max_len,))
        x = k.layers.Embedding(self.config.vocab_size, self.config.embedding_dim)(inputs)
        x = k.layers.Conv1D(self.config.num_filters, self.config.kernel_size, activation='relu')(x)
        x = k.layers.GlobalMaxPooling1D()(x)
        x = k.layers.Dropout(self.config.dropout)(x)
        outputs = k.layers.Dense(self.num_classes, activation='softmax')(x)

        model = k.Model(inputs, outputs)
        model.compile(
            optimizer=k.optimizers.Adam(learning_rate=self.config.learning_rate),
            loss='categorical_crossentropy',
            metrics=['accuracy'],
        )
        model.summary()
        return model

    def _log_model_artifact(self, model):
        os.makedirs('artifacts', exist_ok=True)

        summary_lines = []
        model.summary(print_fn=summary_lines.append)
        with open('artifacts/model_summary.txt', 'w') as f:
            f.write('\n'.join(summary_lines))

        model_path = 'artifacts/20ng_model.keras'
        model.save(model_path)

        art = wandb.Artifact('20ng_classifier', type='model')
        art.add_file('artifacts/model_summary.txt')
        art.add_file(model_path)
        self.run.log_artifact(art)

    def train(self):
        model = self._build_model()

        callbacks = [
            WandbMetricsLogger(log_freq=10),
            WandbModelCheckpoint('checkpoints/model-{epoch:02d}.keras', save_weights_only=False),
            LogLRCallback(),
            LogSamplesCallback(self.X_test, self.y_test, self.test_texts, self.label_names, max_rows=32),
            ConfusionMatrixCallback(self.X_test, self.y_test, self.label_names),
        ]

        model.fit(
            self.X_train, self.y_train,
            validation_data=(self.X_test, self.y_test),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=callbacks,
            verbose=1,
        )

        loss, acc = model.evaluate(self.X_test, self.y_test, verbose=0)
        wandb.log({'final/loss': loss, 'final/accuracy': acc})
        print(f'\nFinal — loss: {loss:.4f}, accuracy: {acc:.4f}')

        self._log_model_artifact(model)
        self.run.finish()

## 5. Run Training

In [8]:
NewsgroupsTrainer().train()

Loading 20 Newsgroups dataset...
Train: (11314, 300), Test: (7532, 300), Classes: 20


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 300, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 296, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,323,668 (5.05 MB)

 Trainable params: 1,323,668 (5.05 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/12
177/177 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - accuracy: 0.1335 - loss: 2.9467 - val_accuracy: 0.3204 - val_loss: 2.7963
Epoch 2/12
177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.4594 - loss: 2.1523 - val_accuracy: 0.5117 - val_loss: 1.7772
Epoch 3/12
177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.6291 - loss: 1.3893 - val_accuracy: 0.5811 - val_loss: 1.4266
Epoch 4/12
177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.7353 - loss: 0.9900 - val_accuracy: 0.6216 - val_loss: 1.2924
Epoch 5/12
177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.8151 - loss: 0.7207 - val_accuracy: 0.6365 - val_loss: 1.2353
Epoch 6/12
177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.8718 - loss: 0.5281 - val_accuracy: 0.6409 - val_loss: 1.2289
Epoch 7/12
177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.9089 - loss: 0.3903 - val_accuracy: 0.6479 - val_loss: 1.2486
Epoch 8/12
177/177 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.9304 - loss: 0.3029 - val_acc

batch/accuracy,▁▂▂▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███████████████
batch/batch_step,▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇█████
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▇▇▆▄▄▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▄▅▆▇▇██████
epoch/epoch,▁▂▂▃▄▄▅▅▆▇▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▄▃▂▂▂▁▁▁▁▁
epoch/val_accuracy,▁▅▆▇████████
epoch/val_loss,█▃▂▁▁▁▁▁▁▁▂▂
+3,...
